## Link al kaggle

https://www.kaggle.com/code/isaidarraga/topicosia-entrega1

CodiEsp

In [3]:
import os

BASE = "/kaggle/input"  # ajusta si hace falta después de ver la salida

print("=== Carpetas y archivos .tsv encontrados ===")
for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith(".tsv"):
            print(os.path.join(root, f))

print("\n=== Primeros archivos .txt encontrados (para confirmar carpeta text_files) ===")
contador = 0
for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith(".txt") and contador < 5:
            print(os.path.join(root, f))
            contador += 1

=== Carpetas y archivos .tsv encontrados ===
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/train/trainP.tsv
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/train/trainX.tsv
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/train/trainD.tsv
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/dev/devP.tsv
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/dev/devX.tsv
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/dev/devD.tsv

=== Primeros archivos .txt encontrados (para confirmar carpeta text_files) ===
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/README.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/text_files_en/S0120-41572009000100005-1.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/

In [4]:
# -*- coding: utf-8 -*-
"""
Filtra casos de piel en CodiEsp usando el código CIE-10 de diagnóstico real
de cada caso -- no palabras sueltas, así que no hay falsos positivos como
"melanoma ocular" o "angioma cerebral" (eso ya lo vimos con SPACCC).

Licencia: CodiEsp es CC BY 4.0 -- se puede reproducir citando:
  Miranda-Escalada A, Gonzalez-Agirre A, Armengol-Estapé J, Krallinger M.
  CodiEsp corpus. Zenodo. https://doi.org/10.5281/zenodo.3758054
"""

import os
import csv
import pandas as pd

BASE = "/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish"
OUTPUT_CSV = "/kaggle/working/codiesp_dermatologia.csv"

# Prefijo del código CIE-10 -> (categoría estilo HAM10000, etiqueta de urgencia)
CODIGOS_OBJETIVO = {
    "c43": ("mel", "urgente"),
    "d03": ("mel", "urgente"),
    "c44": ("bcc", "urgente"),
    "d04": ("akiec", "urgente"),
    "l57": ("akiec", "urgente"),
    "d22": ("nv", "no_urgente"),
    "d23": ("df", "no_urgente"),
    "l82": ("bkl", "no_urgente"),
}


def normalizar_codigo(codigo):
    return str(codigo).strip().lower().replace(".", "")


def match_prefijo(codigo_norm):
    for prefijo, (categoria, urgencia) in CODIGOS_OBJETIVO.items():
        if codigo_norm.startswith(prefijo):
            return categoria, urgencia
    return None, None


def procesar_split(split):
    """split = 'train' o 'dev'"""
    tsv_path = os.path.join(BASE, split, f"{split}D.tsv")
    text_dir = os.path.join(BASE, split, "text_files")

    df = pd.read_csv(tsv_path, sep="\t", header=None, names=["articleID", "codigo"])
    print(f"[{split}] filas en {split}D.tsv: {len(df)}")

    filas = []
    for _, row in df.iterrows():
        codigo_norm = normalizar_codigo(row["codigo"])
        categoria, urgencia = match_prefijo(codigo_norm)
        if categoria is None:
            continue

        ruta_txt = os.path.join(text_dir, f"{row['articleID']}.txt")
        if not os.path.exists(ruta_txt):
            continue

        with open(ruta_txt, encoding="utf-8", errors="ignore") as f:
            texto = f.read().strip().replace("\n", " ")

        filas.append({
            "archivo_origen": row["articleID"],
            "codigo_cie10": row["codigo"],
            "categoria_ham10000": categoria,
            "etiqueta_urgencia": urgencia,
            "texto": texto[:1500],
            "split_original": split,
            "fuente": "CodiEsp (CC BY 4.0) - Miranda-Escalada et al., Zenodo doi:10.5281/zenodo.3758054",
        })

    return filas


def main():
    todas = []
    for split in ["train", "dev"]:
        todas.extend(procesar_split(split))

    print(f"\nTotal de casos de dermatología encontrados: {len(todas)}")

    if todas:
        conteo = pd.DataFrame(todas)["categoria_ham10000"].value_counts()
        print(f"\nDistribución por categoría:\n{conteo}")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "archivo_origen", "codigo_cie10", "categoria_ham10000",
            "etiqueta_urgencia", "texto", "split_original", "fuente"
        ])
        writer.writeheader()
        writer.writerows(todas)

    print(f"\nGuardado en {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

[train] filas en trainD.tsv: 5639
[dev] filas en devD.tsv: 2677

Total de casos de dermatología encontrados: 18

Distribución por categoría:
categoria_ham10000
mel      7
bcc      5
df       2
nv       2
akiec    2
Name: count, dtype: int64

Guardado en /kaggle/working/codiesp_dermatologia.csv


SPACCC

In [5]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files[:3]:
        print(os.path.join(root, f))

/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/README.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/text_files_en/S0120-41572009000100005-1.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/text_files_en/S1135-76062012000300012-1.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/text_files_en/S1139-76322011000300007-1.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/text_files/S0120-41572009000100005-1.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/text_files/S1135-76062012000300012-1.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/test/text_files/S1139-76322011000300007-1.txt
/kaggle/input/datasets/isaidarraga/codiesp-dataset1/final_dataset_v3_to_publish/train/trainP.tsv
/kaggle/input/datasets/isaidarraga/codiesp-data

In [6]:
# -*- coding: utf-8 -*-
"""
Filtra casos clínicos de dermatología del corpus SPACCC (real, CC BY 4.0),
ya descomprimido en una carpeta (no en .zip).

PASO 1: Confirma la ruta exacta de la carpeta "corpus" con el snippet
de os.walk antes de correr esto, y pégala abajo en CORPUS_DIR.

Nota de licencia: SPACCC es CC BY 4.0 — SÍ pueden reproducir estos
textos en su dataset, siempre citando la fuente:
  Intxaurrondo A, et al. SPACCC: Spanish Clinical Case Corpus.
  Zenodo. https://doi.org/10.5281/zenodo.2560316
"""

import os
import csv
import glob

# Ruta confirmada en tu notebook de Kaggle
CORPUS_DIR = "/kaggle/input/datasets/isaidarraga/spaccc-dataset/SPACCC/corpus"
OUTPUT_CSV = "/kaggle/working/spaccc_dermatologia.csv"

# Términos ESPECÍFICOS de nuestras 7 categorías HAM10000 — nada de anatomía
# genérica (piel, dermis, biopsia) porque aparece en cualquier especialidad
# y genera falsos positivos, como ya vimos (292 casos, ninguno de la muestra
# era realmente de dermatología).
TERMINOS_DERMA = [
    "melanoma", "melanocítico", "melanocitico",
    "nevus", "nevo melanocítico", "nevo melanocitico",
    "carcinoma basocelular", "carcinoma basal",
    "carcinoma espinocelular", "carcinoma escamoso cutáneo",
    "queratosis actínica", "queratosis actinica",
    "queratosis seborreica",
    "dermatofibroma",
    "angioma", "hemangioma",
    "lesión pigmentada", "lesion pigmentada",
    "dermatoscopia", "dermatoscopía",
    "lesión cutánea sospechosa", "lesion cutanea sospechosa",
]

TERMINOS_URGENTE = ["melanoma", "carcinoma basocelular", "carcinoma espinocelular",
                     "carcinoma escamoso cutáneo", "queratosis actínica", "maligno"]


def es_de_dermatologia(texto):
    t = texto.lower()
    return any(term in t for term in TERMINOS_DERMA)


def urgencia_probable(texto):
    t = texto.lower()
    return "urgente" if any(term in t for term in TERMINOS_URGENTE) else "revisar_manual"


def main():
    archivos = glob.glob(os.path.join(CORPUS_DIR, "**", "*.txt"), recursive=True)
    print(f"Total de archivos .txt encontrados: {len(archivos)}")

    if len(archivos) == 0:
        print(f"No encontré archivos .txt en {CORPUS_DIR}.")
        print("Corre primero el snippet de os.walk para confirmar la ruta correcta.")
        return

    filas = []
    for ruta in archivos:
        with open(ruta, encoding="utf-8", errors="ignore") as f:
            texto = f.read()
        if es_de_dermatologia(texto):
            filas.append({
                "archivo_origen": os.path.basename(ruta),
                "texto": texto.strip().replace("\n", " ")[:1500],
                "etiqueta_urgencia_sugerida": urgencia_probable(texto),
                "fuente": "SPACCC (CC BY 4.0) - Intxaurrondo et al., Zenodo doi:10.5281/zenodo.2560316",
            })

    print(f"Casos relacionados con dermatología encontrados: {len(filas)}")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "archivo_origen", "texto", "etiqueta_urgencia_sugerida", "fuente"
        ])
        writer.writeheader()
        writer.writerows(filas)

    print(f"Guardado en {OUTPUT_CSV}")
    print("\nIMPORTANTE: revisen cada caso a mano — la etiqueta de urgencia aquí es "
          "solo una primera pasada automática por palabras clave, no un diagnóstico real.")


if __name__ == "__main__":
    main()

Total de archivos .txt encontrados: 1000
Casos relacionados con dermatología encontrados: 34
Guardado en /kaggle/working/spaccc_dermatologia.csv

IMPORTANTE: revisen cada caso a mano — la etiqueta de urgencia aquí es solo una primera pasada automática por palabras clave, no un diagnóstico real.


In [7]:
import pandas as pd

df = pd.read_csv("/kaggle/working/spaccc_dermatologia.csv")

# 1. Exportar el texto completo de cada caso a un .txt legible,
#    para que lo lean sin cortes de 400 caracteres.
with open("/kaggle/working/casos_completos_para_revisar.txt", "w", encoding="utf-8") as f:
    for i, fila in df.iterrows():
        f.write(f"{'='*80}\n")
        f.write(f"CASO {i+1} de {len(df)} — archivo: {fila['archivo_origen']}\n")
        f.write(f"Etiqueta sugerida automática: {fila['etiqueta_urgencia_sugerida']}\n")
        f.write(f"{'='*80}\n")
        f.write(fila['texto'])
        f.write("\n\n")

# 2. Plantilla de revisión: una fila por caso, con columnas para que
#    el equipo llene a mano (repártanse los 34 entre los 4).
plantilla = df[["archivo_origen", "etiqueta_urgencia_sugerida"]].copy()
plantilla["es_realmente_dermatologico"] = ""     # sí / no
plantilla["categoria_ham10000_similar"] = ""     # mel / bcc / akiec / nv / bkl / df / vasc
plantilla["etiqueta_urgencia_final"] = ""        # urgente / no_urgente
plantilla["revisado_por"] = ""                   # nombre de quien lo revisó
plantilla["notas"] = ""

plantilla.to_csv("/kaggle/working/plantilla_revision_equipo.csv", index=False)

print("Archivos generados en /kaggle/working/:")
print("  - casos_completos_para_revisar.txt  (texto completo, sin cortes)")
print("  - plantilla_revision_equipo.csv     (para llenar entre los 4, ~8-9 casos c/u)")
print(f"\nTotal de casos a repartir: {len(df)} (~{len(df)//4} por persona)")

Archivos generados en /kaggle/working/:
  - casos_completos_para_revisar.txt  (texto completo, sin cortes)
  - plantilla_revision_equipo.csv     (para llenar entre los 4, ~8-9 casos c/u)

Total de casos a repartir: 34 (~8 por persona)
